# Continue the original CNN–LSTM 3 experiment — no retraining from zero

**Raj Rishav · IIT Guwahati**

This notebook **does not re-download Kaggle videos**. It mounts the existing Drive subset, checks that the original `best.keras` and seven-label mapping match, then continues from **epoch 3** into a **new** Drive folder. The source checkpoint remains untouched.

The saved optimizer has 36 recorded updates. Exact repeatability of the original interrupted notebook's random-number state is not guaranteed. Test evaluation is separated into an optional last cell and must only be run **once after final model selection**.

Select **Runtime → Change runtime type → GPU** in Colab where available.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import subprocess, sys, os
repo = Path("/content/Raj_Rishav")
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth=1", "https://github.com/Ahmdullah16/Raj_Rishav.git", str(repo)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless", "pandas", "scikit-learn"], check=True)
import tensorflow as tf
print("TensorFlow:", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
from pathlib import Path
data_root = Path("/content/drive/MyDrive/RAJ Rishav/HAR_Kaggle_Subset")
checkpoint = Path("/content/drive/MyDrive/RAJ Rishav/HAR_Results/cnn_lstm_3/best.keras")
class_mapping = repo / "artifacts/classes.json"
split_csv = repo / "data/train_val_test_splits_70_15_15.csv"
output_dir = Path("/content/drive/MyDrive/RAJ Rishav/HAR_Results/cnn_lstm_3_resumed_v1")
for p in (data_root, checkpoint, class_mapping, split_csv):
    if not p.exists():
        raise FileNotFoundError(f"Required existing resource missing: {p}")
print("Existing checkpoint:", checkpoint, "| size:", checkpoint.stat().st_size, "bytes")
print("Source preservation:", checkpoint != output_dir/"best.keras")
print("New results folder:", output_dir)

In [ ]:
os.environ["PYTHONPATH"] = str(repo / "src")
os.environ["HAR_VIDEO_CACHE"] = "/content/har_unpacked_videos"
cmd = [
    sys.executable, "-m", "har.resume",
    "--data-root", str(data_root),
    "--splits", str(split_csv),
    "--resume-from", str(checkpoint),
    "--class-map", str(class_mapping),
    "--initial-epoch", "3",
    "--epochs", "25",
    "--batch-size", "8",
    "--output-dir", str(output_dir),
    "--cache-dir", "/content/har_preprocessed",
    "--max-cache-mb", "256",
]
subprocess.run(cmd, cwd=repo, check=True, env=os.environ.copy())

### After training

Inspect `training_state.json`, `best.keras`, `history.csv`, `classes.json` and `final.keras` in the **new** output folder. If Colab disconnects, rerun the cells above. The runner reuses the checkpoint backup and refuses to pretend completed epochs exist when the backup is absent.

**Do not evaluate the test split to choose hyperparameters or models.** When CNN–LSTM 3 is the final model selected using validation evidence, enable the following cell to evaluate the 20 held-out videos exactly once.

In [ ]:
# Set to True only after final model selection, never during tuning.
RUN_FINAL_TEST_ONCE = False
if RUN_FINAL_TEST_ONCE:
    test_cmd = cmd + ["--evaluate-only"]
    subprocess.run(test_cmd, cwd=repo, check=True, env=os.environ.copy())
else:
    print("Held-out test remains untouched.")